# Student Beer Game: calibration, live progress, and independent games

Use the project `.venv` kernel. Run setup first; opening this notebook makes no API calls. The model is GPT-5.4 Mini's pinned snapshot, with fixed **high** effort and a hypothetical second-year University at Buffalo business-student persona. Weekly calls are not independent students.

The experiment varies total generated tokens, including reasoning. Calibration selects caps using completion and actual token usage, never game outcomes. If no reliable separated pair exists, the saved parameters are labeled diagnostic. A common 180-second timeout is technical, not a human-time treatment.


In [7]:
from pathlib import Path
import json, sys
from IPython.display import display, Markdown
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/beergame_study').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from inside D:/beergame.')
sys.path.insert(0, str(ROOT/'src'))
from beergame_study.reasoning_protocol import ReasoningConfig, StudyStopped
from beergame_study.student_study import run_campaign
from beergame_study.notebook_progress import NotebookProgress, watch_progress
from beergame_study.operational_io import install_windows_file_retry, record_operational_io
install_windows_file_retry()

MODE = 'api'                    # 'offline' is a free SYNTHETIC software check
BUDGET_USD = 10.                 # hard cap for the run you explicitly start
CHAINS_PER_CONDITION = 1         # independent four-role chains; budget may allow fewer
REPLICATE_ID = 'student-notebook-001'  # change for NEW independent games; keep to resume
RUN_CALIBRATION = False
RUN_GAMES = False
WATCH_PROGRESS = True
CORRECT_CALIBRATION_STATES = False   # only for the documented receipt/shipment repair
RECHECK_AFTER_ACCESS_CHANGE = False # enable only after adding credits/fixing limits
RESUME_CALIBRATION = None        # optional compatible interrupted calibration directory

cal_pointer = ROOT/'reports/LATEST_CALIBRATION.json'
cal_info = json.loads(cal_pointer.read_text(encoding='utf-8')) if cal_pointer.exists() else None
CALIBRATION_DIR = Path(cal_info['directory']) if cal_info and cal_info['mode'] == MODE else None
print('Saved calibration:', CALIBRATION_DIR)
if cal_info:
    print('Source mode:', cal_info['mode'], '| manipulation screen passed:', cal_info['manipulation_screen_passed'])
active = ROOT/'reports/ACTIVE_STUDENT_RUN.json'
if active.exists():
    active_info = json.loads(active.read_text(encoding='utf-8'))
    saved_progress = json.loads(Path(active_info['progress']).read_text(encoding='utf-8'))
    NotebookProgress()(saved_progress)


Saved calibration: D:\beergame\data\runs\student_api_b609eca42866feb0\calibration_checkpoint
Source mode: api | manipulation screen passed: False


## Watch the current background run

Set `WATCH_PROGRESS=True` in setup, then run the next cell. It refreshes every two seconds and stops when the run completes or stops. Interrupting this watch cell stops watching only. Runs started in the cells below display their own progress automatically.


In [8]:
if WATCH_PROGRESS:
    await watch_progress(ROOT)
else:
    print('Set WATCH_PROGRESS=True to watch a command-line run live.')


## Secure model client

The key is entered in a hidden prompt and kept in memory. It is never stored in the notebook. No client is created until an explicit run cell is enabled.


In [ ]:
async def execute_study(config, **options):
    import getpass, os
    config.validate()
    install_windows_file_retry()
    if options.get('calibration_run'):
        from beergame_study.student_study import load_calibration
        load_calibration(options['calibration_run'], config)  # validate before asking for a key
    client = None
    if config.mode == 'api':
        from openai import AsyncOpenAI
        key = os.environ.get('OPENAI_API_KEY') or getpass.getpass('OpenAI API key (hidden): ')
        if not key or not key.strip():
            raise RuntimeError('No key supplied; no API calls made.')
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=config.timeout_s)
        key = None
    renderer = NotebookProgress()
    def show_progress(state):
        if not getattr(show_progress, 'io_recorded', False):
            record_operational_io(state['run_directory'], {
                'concurrency': config.concurrency,
                'request_interval_s': config.request_interval_s,
                'entrypoint': 'notebooks/03_student_reasoning_study.ipynb',
            })
            show_progress.io_recorded = True
        renderer(state)
    try:
        directory = await run_campaign(config, client=client, on_progress=show_progress, **options)
        print('Saved run:', directory)
        print('Word report:', directory/'REPORT.docx')
        display(Markdown((directory/'REPORT.md').read_text(encoding='utf-8')))
        return directory
    except StudyStopped as exc:
        print('Controlled stop:', str(exc))
        print('Saved partial report: reports/LATEST_REPORT.txt')
        return None
    finally:
        if client:
            await client.close()


## Optional calibration only

After my calibration finishes, you can skip this section and reuse its saved parameters. To run a new calibration, set `RUN_CALIBRATION=True`. This tests 24 states at six caps (144 decisions), writes the screen and report, and purchases no games. Keep the same replicate ID to resume cached calls. Use a matching `MODE` when reusing calibration; synthetic calibration cannot validate real-model caps.

After restoring credits or limits, you can set `RESUME_CALIBRATION` to the stopped run and `RECHECK_AFTER_ACCESS_CHANGE=True`. The original scientific configuration is retained, and one saved zero-charge 429 may be explicitly rechecked; rejected attempts remain saved. Leave the recheck flag false for ordinary runs.


In [ ]:
if RUN_CALIBRATION:
    cal_config = ReasoningConfig(mode=MODE, stage='game', budget_usd=BUDGET_USD,
        groups_per_condition=CHAINS_PER_CONDITION, replicate_id=REPLICATE_ID+'-calibration',
        concurrency=2, request_interval_s=2.)
    if RESUME_CALIBRATION:
        from dataclasses import replace
        prior = json.loads((Path(RESUME_CALIBRATION)/'manifest.json').read_text(encoding='utf-8'))['config']
        if prior['mode'] != MODE:
            raise ValueError('Resume requires the original mode.')
        cal_config = replace(ReasoningConfig(**prior), budget_usd=BUDGET_USD, concurrency=2, request_interval_s=2.)
    completed_calibration = await execute_study(cal_config, calibration_only=True,
                                               calibration_parent=RESUME_CALIBRATION,
                                               recheck_rejections=RECHECK_AFTER_ACCESS_CHANGE,
                                               correct_calibration_states=CORRECT_CALIBRATION_STATES)
    if completed_calibration:
        CALIBRATION_DIR = completed_calibration
else:
    print('Calibration skipped; saved source:', CALIBRATION_DIR)


## Run additional independent games with saved parameters

Set `RUN_GAMES=True`, choose a **new** `REPLICATE_ID`, and run the next cell. It validates the completed calibration and source code, reuses the caps without buying calibration calls again, and shows a live progress bar. The budget is for this new game run; earlier calibration charges remain separately documented. Up to `CHAINS_PER_CONDITION` pairs are planned from calibration token costs before any game outcomes. The hard guard may stop below the dollar cap.

Each chain has four model roles, 40 weeks, four fixed familiarization weeks, and 12 automatic scoring weeks. Missing answers use flagged mechanical continuation orders, so completed games still need quality checks. Zero orders are valid and no preferred maximum is imposed.


In [ ]:
if RUN_GAMES:
    if CALIBRATION_DIR is None:
        raise RuntimeError('No completed calibration is saved. Run calibration first or set CALIBRATION_DIR.')
    game_config = ReasoningConfig(mode=MODE, stage='game', budget_usd=BUDGET_USD,
        groups_per_condition=CHAINS_PER_CONDITION, replicate_id=REPLICATE_ID,
        concurrency=2, request_interval_s=2.)
    GAME_RUN_DIR = await execute_study(game_config, calibration_run=CALIBRATION_DIR)
else:
    print('Review-only: no games started. Set RUN_GAMES=True when ready.')


## Review the latest report

Reports include completion/fallback checks, costs, bullwhip ratios, actual token usage, and exploratory coefficient-identification diagnostics. Small pilots cannot guarantee significance or establish behavior of real UB students. Validate against real students before that claim. Update README.md and preserve run provenance for every future change.


In [ ]:
pointer = ROOT/'reports/LATEST_REPORT.txt'
paths = [Path(p) for p in pointer.read_text(encoding='utf-8').splitlines()] if pointer.exists() else []
latest_report = next((p for p in paths if p.suffix == '.md' and p.exists()), None)
if latest_report:
    print('Report folder:', latest_report.parent)
    display(Markdown(latest_report.read_text(encoding='utf-8')))
else:
    print('No report yet.')
